In [ ]:
!pip install music21

In [ ]:
import tensorflow as tf
from music21 import corpus, converter, note, chord, stream
import numpy as np
import glob
import os

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("music21 ready")

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
music21 ready


In [ ]:
# Load Bach chorales from music21's built-in corpus
from music21 import corpus

# Get file paths to all Bach chorales
bach_files = corpus.chorales.Iterator()
bach_paths = [c for c in bach_files]

print(f"Total Bach chorales found: {len(bach_paths)}")
print("First file:", bach_paths[0])

Total Bach chorales found: 371
First file: <music21.stream.Score bach/bwv269.mxl>


In [ ]:
from music21 import converter, note, chord
import numpy as np

# We'll use a subset for faster training (first 30 chorales)
NUM_FILES = 30
notes = []

for i, file in enumerate(bach_paths[:NUM_FILES]):
    try:
        midi = file.parse()
        # Get all notes and chords
        notes_to_parse = midi.flatten().notes
        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                # Represent chord as "pitch1.pitch2.pitch3"
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print(f"Skipping file {i}: {e}")

print(f"Total notes/chords extracted: {len(notes)}")
print("First 20:", notes[:20])

Skipping file 0: 'Score' object has no attribute 'parse'
Skipping file 1: 'Score' object has no attribute 'parse'
Skipping file 2: 'Score' object has no attribute 'parse'
Skipping file 3: 'Score' object has no attribute 'parse'
Skipping file 4: 'Score' object has no attribute 'parse'
Skipping file 5: 'Score' object has no attribute 'parse'
Skipping file 6: 'Score' object has no attribute 'parse'
Skipping file 7: 'Score' object has no attribute 'parse'
Skipping file 8: 'Score' object has no attribute 'parse'
Skipping file 9: 'Score' object has no attribute 'parse'
Skipping file 10: 'Score' object has no attribute 'parse'
Skipping file 11: 'Score' object has no attribute 'parse'
Skipping file 12: 'Score' object has no attribute 'parse'
Skipping file 13: 'Score' object has no attribute 'parse'
Skipping file 14: 'Score' object has no attribute 'parse'
Skipping file 15: 'Score' object has no attribute 'parse'
Skipping file 16: 'Score' object has no attribute 'parse'
Skipping file 17: 'Score

In [ ]:
from music21 import note, chord

NUM_FILES = 30
notes = []

for i, midi in enumerate(bach_paths[:NUM_FILES]):
    try:
        notes_to_parse = midi.flatten().notes
        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print(f"Skipping file {i}: {e}")

print(f"Total notes/chords extracted: {len(notes)}")
print("First 20:", notes[:20])

Total notes/chords extracted: 7140
First 20: ['G4', 'D4', 'B3', 'G2', 'G4', 'D4', 'B3', 'G3', 'E4', 'C4', 'E3', 'B3', 'D5', 'D4', 'A3', 'F#3', 'B4', 'D4', 'G3', 'G3']


In [ ]:
# Get unique notes/chords (our vocabulary)
pitchnames = sorted(set(notes))
n_vocab = len(pitchnames)

print(f"Unique notes/chords (vocabulary size): {n_vocab}")
print("Sample vocabulary:", pitchnames[:20])

# Create mapping: note -> integer and integer -> note
note_to_int = {n: i for i, n in enumerate(pitchnames)}
int_to_note = {i: n for i, n in enumerate(pitchnames)}

print("Mappings created.")

Unique notes/chords (vocabulary size): 59
Sample vocabulary: ['A#2', 'A#3', 'A#4', 'A-2', 'A-3', 'A-4', 'A2', 'A3', 'A4', 'A5', 'B#3', 'B-2', 'B-3', 'B-4', 'B2', 'B3', 'B4', 'C#3', 'C#4', 'C#5']
Mappings created.


In [ ]:
# Sequence length = how many notes the model looks at to predict the next one
SEQ_LEN = 50

network_input = []
network_output = []

for i in range(len(notes) - SEQ_LEN):
    seq_in = notes[i:i + SEQ_LEN]
    seq_out = notes[i + SEQ_LEN]
    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

n_patterns = len(network_input)
print(f"Total training patterns: {n_patterns}")
print(f"Input sequence length: {SEQ_LEN}")
print(f"Vocabulary size: {n_vocab}")

Total training patterns: 7090
Input sequence length: 50
Vocabulary size: 59


In [ ]:
from tensorflow.keras.utils import to_categorical
import numpy as np

# Reshape input for LSTM: (samples, timesteps, features)
X = np.reshape(network_input, (n_patterns, SEQ_LEN, 1))
X = X / float(n_vocab)   # normalize

# One-hot encode output
y = to_categorical(network_output)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7090, 50, 1)
y shape: (7090, 59)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Activation
from tensorflow.keras.optimizers import RMSprop

model = Sequential([
    LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(y.shape[1]),
    Activation('softmax')
])

model.compile(loss='categorical_crossentropy',
              optimizer=RMSprop(learning_rate=0.001))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 59)             │        15,163 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 59)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 804,667 (3.07 MB)

 Trainable params: 804,667 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Save the best model during training
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="loss",
    verbose=1,
    save_best_only=True,
    mode="min"
)

EPOCHS = 30
BATCH_SIZE = 128

history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint]
)

Epoch 1/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.6500
Epoch 1: loss improved from None to 3.54482, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 3.5448
Epoch 2/30
53/56 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 3.4943
Epoch 2: loss improved from 3.54482 to 3.48821, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 3.4882
Epoch 3/30
53/56 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 3.4899
Epoch 3: loss improved from 3.48821 to 3.47634, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 3.4763
Epoch 4/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 3.4620
Epoch 4: loss improved from 3.47634 to 3.46699, saving model to best_model.keras

Epoch 4: finished saving model to best_model.keras
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - 

In [ ]:
import random

# Pick a random starting sequence from our data
start = random.randint(0, len(network_input) - 1)
pattern = network_input[start]
prediction_output = []

# Generate 200 new notes
GENERATE_NOTES = 200

print("Generating music...")

for note_index in range(GENERATE_NOTES):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction_input = prediction_input / float(n_vocab)

    prediction = model.predict(prediction_input, verbose=0)
    index = np.argmax(prediction)
    result = int_to_note[index]
    prediction_output.append(result)

    # Slide the window
    pattern.append(index)
    pattern = pattern[1:]

print(f"Generated {len(prediction_output)} notes.")
print("First 20:", prediction_output[:20])

Generating music...
Generated 200 notes.
First 20: ['A4', 'E4', 'E4', 'D4', 'E4', 'A4', 'E4', 'E4', 'D4', 'E4', 'A4', 'E4', 'E4', 'E4', 'A4', 'E4', 'E4', 'E4', 'A4', 'E4']


In [ ]:
from music21 import stream, note, chord
import IPython.display as ipd

offset = 0
output_notes = []

for pattern in prediction_output:
    # If it's a chord (like "0.4.7")
    if ('.' in pattern) or pattern.isdigit():
        notes_in_chord = pattern.split('.')
        chord_notes = []
        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = None
            chord_notes.append(new_note)
        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)
    # Otherwise it's a single note
    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        output_notes.append(new_note)

    # Advance the offset by 0.5 beats
    offset += 0.5

# Create MIDI stream
midi_stream = stream.Stream(output_notes)

# Save to file
output_file = "generated_music.mid"
midi_stream.write('midi', fp=output_file)

print(f"MIDI file saved: {output_file}")

# Play in Colab
print("Playing generated music...")
ipd.display(ipd.Audio(output_file))

MIDI file saved: generated_music.mid
Playing generated music...


In [ ]:
from google.colab import files

files.download("generated_music.mid")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>